<!-- parity-note -->
## MATLAB Parity Note
- Source MATLAB helpfile: `HybridFilterExample.mlx`
- Fidelity status: `high_fidelity`
- Remaining justified differences: The notebook now reproduces the hybrid-filter simulation, single-run decoding, and averaged summary figures with real outputs; the Python port still uses the current hybrid-filter implementation instead of every MATLAB-specific reporting branch.


In [ ]:
# nSTAT-python notebook example: HybridFilterExample
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SRC_PATH = (REPO_ROOT / "src").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from nstat.notebook_figures import FigureTracker
from nstat.paper_examples_full import run_experiment6

np.random.seed(0)
OUTPUT_ROOT = REPO_ROOT / "output" / "notebook_images"
__tracker = FigureTracker(topic='HybridFilterExample', output_root=OUTPUT_ROOT, expected_count=3)


def _prepare_figure(matlab_line: str, *, figsize=(8.0, 4.5)):
    fig = __tracker.new_figure(matlab_line)
    fig.clear()
    fig.set_size_inches(*figsize)
    return fig


def _plot_raster(ax, time_s, spikes, *, max_cells=18):
    n_cells = min(int(spikes.shape[1]), max_cells)
    for row in range(n_cells):
        spike_times = np.asarray(time_s, dtype=float)[np.asarray(spikes[:, row], dtype=float) > 0.5]
        if spike_times.size:
            ax.vlines(spike_times, row + 0.6, row + 1.4, color="k", linewidth=0.35)
    ax.set_ylim(0.5, n_cells + 0.5)
    ax.set_ylabel("cell")


In [ ]:
# SECTION 0: Hybrid Point Process Filter Example
# This notebook mirrors the MATLAB hybrid-filter helpfile with executable figures.
plt.close("all")
summary, payload = run_experiment6(REPO_ROOT, return_payload=True)
batch_payloads = [run_experiment6(REPO_ROOT, seed=37 + idx, return_payload=True)[1] for idx in range(4)]
mean_state_prob_2 = np.mean([row["state_prob_2"] for row in batch_payloads], axis=0)
mean_decoded_x = np.mean([row["decoded_x"] for row in batch_payloads], axis=0)
mean_decoded_y = np.mean([row["decoded_y"] for row in batch_payloads], axis=0)
print(
    {
        "num_samples": int(summary["num_samples"]),
        "num_cells": int(summary["num_cells"]),
        "state_accuracy": round(float(summary["state_accuracy"]), 3),
    }
)


In [ ]:
# SECTION 1: Problem Statement
# We infer both a discrete movement state and a continuous reach trajectory from point-process observations.
pass


In [ ]:
# SECTION 2: Hybrid state-space setup
# The Python port keeps the same two-state problem structure as MATLAB: a low-motion state and a movement state.
pass


In [ ]:
# SECTION 3: Generated Simulated Arm Reach
fig = _prepare_figure("fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...", figsize=(10.0, 9.0))
axs = fig.subplots(4, 2)
axs[0, 0].plot(100.0 * payload["x_pos"], 100.0 * payload["y_pos"], color="k", linewidth=1.8)
axs[0, 0].scatter([100.0 * payload["x_pos"][0]], [100.0 * payload["y_pos"][0]], color="tab:blue", s=35, label="Start")
axs[0, 0].scatter([100.0 * payload["x_pos"][-1]], [100.0 * payload["y_pos"][-1]], color="tab:red", s=35, label="Finish")
axs[0, 0].set_title("Reach path")
axs[0, 0].set_xlabel("X [cm]")
axs[0, 0].set_ylabel("Y [cm]")
axs[0, 0].legend(loc="best", frameon=False, fontsize=8)
_plot_raster(axs[0, 1], payload["time_s"], payload["spikes"])
axs[0, 1].set_title("Neural raster")
axs[1, 0].plot(payload["time_s"], payload["state_true"], color="k", linewidth=1.8)
axs[1, 0].set_yticks([1, 2], ["N", "M"])
axs[1, 0].set_title("Discrete movement state")
axs[1, 1].plot(payload["time_s"], 100.0 * payload["x_pos"], color="tab:blue", linewidth=1.3, label="x")
axs[1, 1].plot(payload["time_s"], 100.0 * payload["y_pos"], color="tab:orange", linewidth=1.3, label="y")
axs[1, 1].set_title("Position")
axs[1, 1].legend(loc="best", frameon=False, fontsize=8)
axs[2, 0].plot(payload["time_s"], 100.0 * payload["x_vel"], color="tab:blue", linewidth=1.3, label="vx")
axs[2, 0].plot(payload["time_s"], 100.0 * payload["y_vel"], color="tab:orange", linewidth=1.3, label="vy")
axs[2, 0].set_title("Velocity")
axs[2, 0].legend(loc="best", frameon=False, fontsize=8)
axs[2, 1].plot(payload["time_s"], np.mean(payload["spikes"], axis=1), color="tab:green", linewidth=1.2)
axs[2, 1].set_title("Population spike fraction")
axs[3, 0].plot(payload["time_s"], np.cumsum(payload["spikes"], axis=0)[:, 0], color="tab:purple", linewidth=1.1)
axs[3, 0].set_title("Example cumulative spike count")
axs[3, 1].axis("off")
axs[3, 1].text(
    0.0,
    0.95,
    "\n".join(
        [
            f"Cells: {int(summary['num_cells'])}",
            f"State accuracy: {summary['state_accuracy']:.3f}",
            f"Decode RMSE X: {summary['decode_rmse_x']:.3f}",
            f"Decode RMSE Y: {summary['decode_rmse_y']:.3f}",
        ]
    ),
    va="top",
    family="monospace",
    fontsize=9,
)


In [ ]:
# SECTION 4: Simulate Neural Firing
# The simulated spike population depends on the latent state and the movement dynamics.
pass


In [ ]:
# SECTION 5: Run the hybrid filter
fig = _prepare_figure("subplot(4,3,[1 4])", figsize=(11.0, 9.0))
axs = fig.subplots(4, 3)
decoded_vx = np.gradient(payload["decoded_x"], payload["time_s"])
decoded_vy = np.gradient(payload["decoded_y"], payload["time_s"])
axs[0, 0].plot(payload["time_s"], payload["state_true"], color="k", linewidth=1.8, label="True")
axs[0, 0].plot(payload["time_s"], payload["state_hat"], color="tab:blue", linewidth=1.0, label="Estimated")
axs[0, 0].set_yticks([1, 2], ["N", "M"])
axs[0, 0].set_title("State estimate")
axs[0, 0].legend(loc="best", frameon=False, fontsize=8)
axs[0, 1].plot(payload["time_s"], payload["state_prob_2"], color="tab:blue", linewidth=1.2)
axs[0, 1].set_title("Pr(Movement)")
axs[0, 2].plot(100.0 * payload["x_pos"], 100.0 * payload["y_pos"], color="k", linewidth=1.6, label="True")
axs[0, 2].plot(100.0 * payload["decoded_x"], 100.0 * payload["decoded_y"], color="tab:blue", linewidth=1.2, label="Decoded")
axs[0, 2].set_title("Movement path")
axs[0, 2].legend(loc="best", frameon=False, fontsize=8)
axs[1, 0].plot(payload["time_s"], 100.0 * payload["x_pos"], color="k", linewidth=1.6)
axs[1, 0].plot(payload["time_s"], 100.0 * payload["decoded_x"], color="tab:blue", linewidth=1.2)
axs[1, 0].set_title("X position")
axs[1, 1].plot(payload["time_s"], 100.0 * payload["y_pos"], color="k", linewidth=1.6)
axs[1, 1].plot(payload["time_s"], 100.0 * payload["decoded_y"], color="tab:blue", linewidth=1.2)
axs[1, 1].set_title("Y position")
axs[1, 2].plot(payload["time_s"], 100.0 * payload["x_vel"], color="k", linewidth=1.6)
axs[1, 2].plot(payload["time_s"], 100.0 * decoded_vx, color="tab:blue", linewidth=1.2)
axs[1, 2].set_title("X velocity")
axs[2, 0].plot(payload["time_s"], 100.0 * payload["y_vel"], color="k", linewidth=1.6)
axs[2, 0].plot(payload["time_s"], 100.0 * decoded_vy, color="tab:blue", linewidth=1.2)
axs[2, 0].set_title("Y velocity")
axs[2, 1].plot(payload["time_s"], np.sqrt((payload["decoded_x"] - payload["x_pos"]) ** 2 + (payload["decoded_y"] - payload["y_pos"]) ** 2), color="tab:red", linewidth=1.2)
axs[2, 1].set_title("Instantaneous path error")
axs[2, 2].hist(np.sum(payload["spikes"], axis=0), bins=12, color="tab:green", alpha=0.85)
axs[2, 2].set_title("Spike counts per cell")
axs[3, 0].axis("off")
axs[3, 1].axis("off")
axs[3, 2].axis("off")

fig = _prepare_figure("plot(time,mean(S_estAll))", figsize=(10.0, 7.0))
axs = fig.subplots(2, 2)
axs[0, 0].plot(payload["time_s"], payload["state_true"], color="k", linewidth=1.6, label="True state")
axs[0, 0].plot(payload["time_s"], 1.0 + (mean_state_prob_2 > 0.5).astype(float), color="tab:blue", linewidth=1.2, label="Mean estimate")
axs[0, 0].set_yticks([1, 2], ["N", "M"])
axs[0, 0].legend(loc="best", frameon=False, fontsize=8)
axs[0, 0].set_title("Average state estimate")
axs[0, 1].plot(payload["time_s"], mean_state_prob_2, color="tab:blue", linewidth=1.2)
axs[0, 1].set_title("Average Pr(Movement)")
axs[1, 0].plot(100.0 * payload["x_pos"], 100.0 * payload["y_pos"], color="k", linewidth=1.6, label="True")
axs[1, 0].plot(100.0 * mean_decoded_x, 100.0 * mean_decoded_y, color="tab:blue", linewidth=1.2, label="Mean decoded")
axs[1, 0].legend(loc="best", frameon=False, fontsize=8)
axs[1, 0].set_title("Average decoded path")
axs[1, 1].bar(
    ["X RMSE", "Y RMSE"],
    [summary["decode_rmse_x"], summary["decode_rmse_y"]],
    color=["tab:blue", "tab:orange"],
)
axs[1, 1].set_title("Single-run decoding RMSE")
__tracker.finalize()
